# AxiomAI - LLM Router Training Notebook

This notebook trains a DistilBERT model to classify prompts as requiring a `small_llm` or `large_llm`.

In [1]:
# Cell 1: Install + import libraries
import pandas as pd
import numpy as np

# Hugging Face
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# ML metrics
from sklearn.metrics import accuracy_score, f1_score

print("All libraries imported successfully!")

C:\Projects\Software Projects\AxiomAI\venv_backend\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


In [2]:
# Cell 2: Load the HuggingFace dataset
dataset = load_dataset("DevQuasar/llm_router_dataset-synth")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'label'],
        num_rows: 15306
    })
    test: Dataset({
        features: ['id', 'prompt', 'label'],
        num_rows: 4921
    })
})

In [3]:
# Cell 3: Look at the data
dataset["train"][0]

{'id': '50bc8bed-b7be-436c-9439-9cf62555765b',
 'prompt': 'What are the implications of quantum entanglement on modern cryptography?',
 'label': 1}

In [4]:
# Cell 4: Convert labels to numbers
def label_to_int(label):
    if label == "small_llm":
        return 0
    else:
        return 1

dataset = dataset.map(lambda x: {
    "text": x["prompt"],
    "label": label_to_int(x["label"])
})

# Check the transformation
dataset["train"][0]

Map: 100%|███████████████████████████████████████████████████████████████| 4921/4921 [00:00<00:00, 15963.13 examples/s]


{'id': '50bc8bed-b7be-436c-9439-9cf62555765b',
 'prompt': 'What are the implications of quantum entanglement on modern cryptography?',
 'label': 1,
 'text': 'What are the implications of quantum entanglement on modern cryptography?'}

In [5]:
# Cell 5: Train/validation split
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'label', 'text'],
        num_rows: 12244
    })
    test: Dataset({
        features: ['id', 'prompt', 'label', 'text'],
        num_rows: 3062
    })
})

In [6]:
# Cell 6: Tokenization
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize, batched=True)
print("Tokenization complete!")

Map: 100%|████████████████████████████████████████████████████████████████| 3062/3062 [00:00<00:00, 9031.55 examples/s]

Tokenization complete!


In [7]:
# Cell 7: Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
print("Model loaded!")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded!


In [11]:
# Cell 8: Define metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

In [9]:
# Cell 9: Training configuration
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

In [20]:
# Cell 10: Train the model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [19]:
# Cell 11: Save the trained model
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")
print("Model saved to ./saved_model")

Model saved to ./saved_model
